# Drop All User Schemas and Tables

This notebook completely removes all user schemas and tables.
- Automatically discovers all user schemas and tables
- Skips system schemas
- ⚠️ **WARNING**: This permanently deletes ALL user data and structures!

In [ ]:
# === DROP ALL USER DATA ===
def drop_all_user_data():
    """Drop all user schemas and tables (excludes system schemas)"""
    
    try:
        # Get all schemas, excluding system ones
        all_schemas = spark.sql("SHOW DATABASES").collect()
        system_schemas = {'default', 'information_schema', 'sys', 'hive_metastore'}
        user_schemas = [schema[0] for schema in all_schemas if schema[0] not in system_schemas]
        
        if not user_schemas:
            print("✅ No user schemas found!")
            return 0
        
        print(f"🔍 Found schemas: {', '.join(user_schemas)}")
        print("🚨 WARNING: This will permanently delete ALL user data!")
        
        total_dropped = 0
        # Step 1: Drop all tables in each schema
        for schema in user_schemas:
            try:
                # Get all tables in schema
                tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
                table_names = [table.tableName for table in tables]
                
                if not table_names:
                    print(f"📭 No tables in {schema}")
                else:
                    print(f"\n🗑️ Dropping {len(table_names)} tables in '{schema}'...")
                    
                    for table_name in table_names:
                        full_table = f"{schema}.{table_name}"
                        try:
                            spark.sql(f"DROP TABLE IF EXISTS {full_table}")
                            print(f"  ✅ {full_table}")
                            total_dropped += 1
                        except Exception as e:
                            print(f"  ❌ {full_table}: {e}")
                            
            except Exception as e:
                print(f"⚠️ Error processing schema {schema}: {e}")
        
        # Step 2: Drop empty schemas
        print(f"\n🗑️ Dropping empty schemas...")
        for schema in user_schemas:
            try:
                spark.sql(f"DROP DATABASE IF EXISTS {schema} CASCADE")
                print(f"  ✅ Schema {schema}")
            except Exception as e:
                print(f"  ❌ Schema {schema}: {e}")
        
        print(f"\n🎉 Completed! {total_dropped} tables and {len(user_schemas)} schemas processed.")
        return total_dropped
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return 0

# === EXECUTE ===
print("🚨 DROPPING ALL USER SCHEMAS AND TABLES")
drop_all_user_data()
print("💀 ALL USER DATA AND STRUCTURES REMOVED!")